# 12: Custom Firmware Integration

## Objective
Add custom Verilog/VHDL modules, integrate with AXI-lite interface, and rebuild the firmware.

## Prerequisites
- Vivado installation (2020.1 or later)
- QICK source code
- Basic Verilog/VHDL knowledge

In [ ]:
import os
from qick import QickSoc

# Check current firmware info
soc = QickSoc(bitfile="/path/to/current.bit")
cfg = soc.get_cfg()

print(f"Current firmware version: {cfg['fw_version']}")
print(f"Build date: {cfg.get('build_date', 'Unknown')}")
print(f"Git hash: {cfg.get('git_hash', 'Unknown')}")

## Custom Verilog Module Example

Create a file `custom_module.v`:

```verilog
module custom_module #(
    parameter AXI_ADDR_WIDTH = 32,
    parameter AXI_DATA_WIDTH = 32
)(
    input wire clk,
    input wire rst,
    
    // AXI-lite slave interface
    input wire [AXI_ADDR_WIDTH-1:0] s_axi_awaddr,
    input wire s_axi_awvalid,
    output wire s_axi_awready,
    // ... (full AXI interface)
    
    // Custom output
    output reg [31:0] custom_output
);

    reg [31:0] control_reg;
    
    always @(posedge clk) begin
        if (rst) begin
            control_reg <= 32'd0;
            custom_output <= 32'd0;
        end else begin
            // AXI write logic here
            custom_output <= control_reg;
        end
    end

endmodule
```

## Integration Steps

### Step 1: Add module to QICK source
```bash
cp custom_module.v $QICK_PATH/firmware/ip/
```

### Step 2: Update block design
```tcl
# In Vivado TCL console
create_bd_cell -type ip -vlnv user:user:custom_module:1.0 custom_module_0
connect_bd_intf_net -intf_net axi_interconnect_0_m00_axi [get_bd_intf_pins custom_module_0/s_axi]
```

### Step 3: Rebuild firmware
```bash
cd $QICK_PATH/firmware
make clean
make all
```

In [ ]:
# Python interface to custom module
class CustomModule:
    def __init__(self, soc, base_addr=0x40000000):
        self.soc = soc
        self.base = base_addr
    
    def write_reg(self, offset, value):
        self.soc.axilite_write(self.base + offset, value)
    
    def read_reg(self, offset):
        return self.soc.axilite_read(self.base + offset)
    
    def set_output(self, value):
        self.write_reg(0x00, value)
    
    def get_status(self):
        return self.read_reg(0x04)

# Usage example
# custom = CustomModule(soc)
# custom.set_output(0xDEADBEEF)
# print(f"Status: {custom.get_status():08X}")

## Debugging Tips

1. **Check AXI connectivity:**
```bash
# In Vivado
report_ip_status
validate_bd_design
```

2. **Monitor signals in hardware:**
```python
# Add ILA core to your design
# Then read captured data
ila_data = soc.axilite_read(0x43C00000, length=1024)
```

3. **Common issues:**
- Clock crossing violations
- AXI address mapping conflicts
- Timing closure failures

## Resources
- [QICK Firmware Documentation](https://qick.readthedocs.io/en/latest/firmware.html)
- [Vivado AXI Reference Guide](https://www.xilinx.com/support/documentation/ip_documentation/axi_ref_guide/latest/ug1037-vivado-axi-reference-guide.pdf)

In [ ]:
# Quick test after firmware update
def verify_custom_firmware(soc, expected_version="1.0"):
    """Verify custom firmware is loaded correctly"""
    try:
        version_reg = soc.axilite_read(0x40000000 + 0xFC)
        version = f"{(version_reg >> 16) & 0xFFFF}.{(version_reg >> 8) & 0xFF}"
        print(f"Custom firmware version: {version}")
        return version == expected_version
    except:
        print("Custom module not detected")
        return False

# verify_custom_firmware(soc)